# Semana 2: Tensores - La Estrella del Data Science
## Notebook Conceptual (NB1) - Manipulación de Tensores con Datos Sintéticos

### Propósito de la sesión
Dominar la estructura de datos fundamental en IA: el **tensor**. Entenderemos cómo se organiza la información en diferentes dominios (como visión por computador y procesamiento de lenguaje natural) y realizaremos operaciones clave de manipulación.

### Objetivos de aprendizaje
*   Definir y diferenciar escalares, vectores, matrices y tensores de alto orden.
*   Crear y manipular tensores usando **NumPy** (y una introducción a **PyTorch**).
*   Realizar operaciones fundamentales: **reshape**, **transposición** y **slicing**.
*   Calcular e interpretar las normas **L1**, **L2** y **L-infinito**.
*   Conectar estas estructuras y operaciones con aplicaciones reales en Machine Learning, Deep Learning, CV y NLP.

## Configuración Inicial

Importamos las librerías necesarias. Además de NumPy, exploraremos PyTorch, el framework de deep learning que usa tensores de forma nativa.

In [ ]:
# Importamos librerías estándar
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Intentamos importar PyTorch. Si no está instalado, se instalará en Google Colab.
try:
    import torch
    print("✅ PyTorch importado correctamente.")
except ImportError:
    print("❌ PyTorch no encontrado. Instalando...")
    !pip install torch
    import torch
    print("✅ PyTorch instalado e importado.")

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("\n🎯 Librerías listas para trabajar con tensores.")

---
## 1. ¿Qué es un Tensor? Del Escalar al Tensor 4D

Un **tensor** es un contenedor multidimensional de números. Es la generalización de matrices a más de dos dimensiones. El **orden** (o rango) de un tensor es el número de dimensiones que tiene.

### 1.1. Creación de Tensores con NumPy y PyTorch

Crearemos tensores de orden 0 a 4 para entender cómo aumenta la complejidad.

In [ ]:
# --- Tensor de Orden 0: Escalar ---
# Un número simple. En IA, puede ser un bias, una temperatura, etc.
escalar_np = np.array(5.2)
escalar_pt = torch.tensor(5.2)

print("🔷 Tensor de Orden 0 (Escalar)")
print(f"  NumPy: {escalar_np}, tipo: {type(escalar_np)}, dimensiones: {escalar_np.ndim}, forma: {escalar_np.shape}")
print(f"  PyTorch: {escalar_pt}, tipo: {type(escalar_pt)}, dimensiones: {escalar_pt.dim()}, forma: {escalar_pt.shape}\n")

# --- Tensor de Orden 1: Vector ---
# Una lista ordenada. Representa las características de una instancia.
vector_np = np.array([1.5, 2.0, -0.5, 3.2])
vector_pt = torch.tensor([1.5, 2.0, -0.5, 3.2])

print("🔶 Tensor de Orden 1 (Vector)")
print(f"  NumPy: {vector_np}, forma: {vector_np.shape}")
print(f"  PyTorch: {vector_pt}, forma: {vector_pt.shape}\n")

# --- Tensor de Orden 2: Matriz ---
# Una tabla bidimensional. Una imagen en escala de grises es una matriz.
matriz_np = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
matriz_pt = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print("🔷 Tensor de Orden 2 (Matriz)")
print(f"  NumPy:\n{matriz_np}\n  Forma: {matriz_np.shape}")
print(f"  PyTorch:\n{matriz_pt}\n  Forma: {matriz_pt.shape}\n")

# --- Tensor de Orden 3 ---
# Una imagen a color (alto, ancho, canales) o un lote de vectores.
# Ejemplo: 2 imágenes en escala de grises de 4x4 píxeles.
tensor_3d_np = np.random.rand(2, 4, 4)  # 2 imágenes, 4x4
tensor_3d_pt = torch.randn(2, 4, 4)

print("🔶 Tensor de Orden 3 (Ej: lote de imágenes en escala de grises)")
print(f"  NumPy forma: {tensor_3d_np.shape}")
print(f"  PyTorch forma: {tensor_3d_pt.shape}\n")

# --- Tensor de Orden 4 ---
# Un lote de imágenes a color (batch, alto, ancho, canales).
# Ejemplo: 3 imágenes RGB de 32x32 píxeles.
tensor_4d_np = np.random.rand(3, 32, 32, 3)
tensor_4d_pt = torch.randn(3, 32, 32, 3)

print("🎯 Tensor de Orden 4 (Ej: lote de imágenes a color)")
print(f"  NumPy forma: {tensor_4d_np.shape}")
print(f"  PyTorch forma: {tensor_4d_pt.shape}")

### 1.2. Visualización de Tensores

Visualizar tensores de más de 2D es un desafío, pero podemos representar sus dimensiones y "rebanadas" para entender su estructura.

In [ ]:
# Función para dibujar un tensor 3D como un cubo de matrices
def plot_tensor_3d(tensor, title="Tensor 3D (Cubo de Matrices)"):
    if tensor.ndim != 3:
        print("Esta función es solo para tensores 3D.")
        return
    fig = plt.figure(figsize=(12, 5))

    # Mostramos las matrices a lo largo de la primera dimensión
    num_mats = min(tensor.shape[0], 5)  # Mostrar máximo 5 para no saturar
    for i in range(num_mats):
        ax = fig.add_subplot(1, num_mats, i+1)
        ax.imshow(tensor[i, :, :], cmap='viridis', aspect='auto')
        ax.set_title(f'Índice {i}')
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Creamos un tensor 3D con patrones para visualizar mejor
patron = np.array([[i+j for j in range(5)] for i in range(5)])
tensor_viz = np.array([patron + i*10 for i in range(4)])
print(f"Tensor 3D de forma {tensor_viz.shape} (4 matrices de 5x5)")
plot_tensor_3d(tensor_viz, title="Tensor 3D: 4 capas de matrices 5x5")

---
## 2. Operaciones Fundamentales con Tensores

### 2.1. Redimensionamiento (Reshape)
Cambiar la forma de un tensor sin alterar los datos. Es crucial para alimentar datos a modelos (ej. aplanar una imagen para una capa fully-connected).

In [ ]:
# Vector a Matriz
v = np.array([1, 2, 3, 4, 5, 6])
print(f"Vector original: {v}, forma: {v.shape}")

m_2x3 = v.reshape(2, 3)
print(f"Reshape a matriz 2x3:\n{m_2x3}")

m_3x2 = v.reshape(3, 2)
print(f"Reshape a matriz 3x2:\n{m_3x2}\n")

# Matriz a Tensor 3D
m = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])
print(f"Matriz original:\n{m}, forma: {m.shape}")

t_2x2x2 = m.reshape(2, 2, 2)  # 2 matrices de 2x2
print(f"Reshape a tensor 3D (2x2x2):\n{t_2x2x2}\n")

# Aplanar (flatten) una matriz a vector
vector_plano = m.reshape(-1)  # -1 le dice a NumPy que infiera la dimensión
print(f"Matriz aplanada a vector: {vector_plano}, forma: {vector_plano.shape}")

# Uso del -1 para inferir dimensiones
otra_forma = v.reshape(2, -1)  # 2 filas, columnas inferidas
print(f"Uso de -1 para inferir columnas: {otra_forma.shape}")

# ¡Importante! El número total de elementos debe coincidir.
try:
    v.reshape(2, 4)
except ValueError as e:
    print(f"\n❌ Error esperado: {e}")

### 2.2. Transposición
Intercambiar dimensiones. Para matrices, es el intercambio de filas y columnas. Para tensores de alto orden, podemos permutar ejes arbitrariamente.

In [ ]:
# Transposición de una matriz
A = np.array([[1, 2, 3], [4, 5, 6]])
print(f"Matriz A (2x3):\n{A}")
print(f"Transpuesta A.T (3x2):\n{A.T}\n")

# Transposición de un tensor 3D
# Supongamos un tensor que representa imágenes: (batch, alto, ancho)
imagenes = np.random.rand(2, 4, 5)  # 2 imágenes de 4x5
print(f"Tensor original (batch, alto, ancho): {imagenes.shape}")

# Queremos permutar a (alto, ancho, batch) para alguna operación
permutado = np.transpose(imagenes, axes=(1, 2, 0))
print(f"Tensor transpuesto (alto, ancho, batch): {permutado.shape}\n")

# En PyTorch, usamos permute()
if torch.cuda.is_available():
    imagenes_pt = torch.randn(2, 4, 5)
    permutado_pt = imagenes_pt.permute(1, 2, 0)
    print(f"PyTorch - Original: {imagenes_pt.shape}, Permutado: {permutado_pt.shape}")

### 2.3. Slicing (Indexación y Rebanado)
Extraer subconjuntos de tensores. Es una habilidad fundamental para acceder a lotes, canales o regiones específicas.

In [ ]:
# Creamos un tensor 3D que simula un lote de 3 imágenes RGB de 5x5
# Forma: (batch=3, alto=5, ancho=5, canales=3)
lote_imagenes = np.random.randint(0, 256, size=(3, 5, 5, 3), dtype=np.uint8)
print(f"Forma del lote: {lote_imagenes.shape}\n")

# 1. Seleccionar la primera imagen del lote (batch index 0)
img_0 = lote_imagenes[0, :, :, :]
print(f"Primera imagen - forma: {img_0.shape}")

# 2. Seleccionar el canal rojo (canal 0) de todas las imágenes
canal_rojo = lote_imagenes[:, :, :, 0]
print(f"Canal rojo de todo el lote - forma: {canal_rojo.shape}")

# 3. Seleccionar un parche (patch) de 2x2 de la primera imagen, solo del canal verde (canal 1)
patch = lote_imagenes[0, 1:3, 2:4, 1]
print(f"Parche 2x2 (fila 1-2, col 2-3) del canal verde de la primera imagen - forma: {patch.shape}\n")

# 4. Seleccionar todas las imágenes, pero solo las filas pares y columnas impares, todos los canales
sub_muestreo = lote_imagenes[:, ::2, 1::2, :]
print(f"Sub-muestreo (filas pares, columnas impares) - forma: {sub_muestreo.shape}")

# Visualización del slicing: mostramos el parche extraído como una imagen diminuta
plt.imshow(patch, cmap='Greens', vmin=0, vmax=255)
plt.title("Parche extraído (canal verde)")
plt.colorbar()
plt.show()

---
## 3. Normas de Vectores: Midiendo la Magnitud

Una **norma** es una función que asigna a un vector su longitud o magnitud. Son esenciales en regularización y en muchas métricas.

### 3.1. Norma L2 (Euclidiana)
$$\|\mathbf{x}\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2}$$
Mide la distancia en línea recta desde el origen.

### 3.2. Norma L1 (Manhattan)
$$\|\mathbf{x}\|_1 = \sum_{i=1}^{n} |x_i|$$
Suma de valores absolutos. Es más robusta a valores atípicos.

### 3.3. Norma L-Infinito
$$\|\mathbf{x}\|_\infty = \max_{i} |x_i|$$
La magnitud del elemento más grande.

In [ ]:
# Vector de ejemplo
v = np.array([3, -4])
print(f"Vector v: {v}")

# Cálculo de normas con NumPy
norma_l2 = np.linalg.norm(v, ord=2)
norma_l1 = np.linalg.norm(v, ord=1)
norma_linf = np.linalg.norm(v, ord=np.inf)

print(f"Norma L2 (euclidiana): {norma_l2:.2f}")
print(f"Norma L1 (Manhattan): {norma_l1:.2f}")
print(f"Norma L-infinito: {norma_linf:.2f}")

# Verificación manual
l2_manual = np.sqrt(3**2 + (-4)**2)
l1_manual = np.abs(3) + np.abs(-4)
linf_manual = np.max(np.abs([3, -4]))
print(f"\nVerificación manual - L2: {l2_manual}, L1: {l1_manual}, L-inf: {linf_manual}")

### 3.4. Visualización de las Normas
Las "bolas unitarias" (conjunto de puntos con norma = 1) para diferentes normas tienen formas características.

In [ ]:
# Creamos una malla de puntos en 2D
x = np.linspace(-1.5, 1.5, 200)
y = np.linspace(-1.5, 1.5, 200)
X, Y = np.meshgrid(x, y)

# Calculamos las normas para cada punto
norma_l2 = np.sqrt(X**2 + Y**2)
norma_l1 = np.abs(X) + np.abs(Y)
norma_linf = np.maximum(np.abs(X), np.abs(Y))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Norma L2 (Círculo)
contour_l2 = axes[0].contour(X, Y, norma_l2, levels=[1], colors='blue', linewidths=2)
axes[0].set_title('Bola Unitaria L2 (Círculo)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Norma L1 (Rombo/Diamante)
contour_l1 = axes[1].contour(X, Y, norma_l1, levels=[1], colors='red', linewidths=2)
axes[1].set_title('Bola Unitaria L1 (Rombo)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

# Norma L-infinito (Cuadrado)
contour_linf = axes[2].contour(X, Y, norma_linf, levels=[1], colors='green', linewidths=2)
axes[2].set_title('Bola Unitaria L-∞ (Cuadrado)')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
axes[2].grid(True, alpha=0.3)
axes[2].axis('equal')

plt.tight_layout()
plt.suptitle('Formas de las Bolas Unitarias para Diferentes Normas', y=1.05, fontsize=14)
plt.show()

---
## 4. Conexiones IA: Aplicaciones Reales

Ahora conectaremos todo lo aprendido con aplicaciones prácticas en IA.

### 4.1. Visión por Computador (CV)
*   Una **imagen en color** es un tensor 3D: (alto, ancho, canales).
*   Un **lote de imágenes** es un tensor 4D: (batch, alto, ancho, canales).

In [ ]:
# Simulamos un lote de 2 imágenes RGB de 64x64
batch_size, height, width, channels = 2, 64, 64, 3
lote_cv = np.random.rand(batch_size, height, width, channels)
print(f"🎥 Tensor CV (lote de imágenes): {lote_cv.shape}")

# Extraemos y visualizamos la primera imagen (solo el primer canal para escala de grises)
primera_imagen_canal0 = lote_cv[0, :, :, 0]
plt.imshow(primera_imagen_canal0, cmap='gray')
plt.title("Primera imagen del lote (Canal 0 - simulado)")
plt.axis('off')
plt.show()

### 4.2. Procesamiento de Lenguaje Natural (NLP)
*   Un **embedding de palabra** es un vector (tensor 1D).
*   Una **oración** (secuencia de palabras) es una matriz (tensor 2D).
*   Un **lote de oraciones** es un tensor 3D: (batch, secuencia, dimensión_embedding).

In [ ]:
# Parámetros típicos
batch_size = 4
seq_length = 10   # 10 palabras por oración
embedding_dim = 300  # Dimensión del embedding (ej. GloVe)

lote_nlp = np.random.randn(batch_size, seq_length, embedding_dim)
print(f"📝 Tensor NLP (lote de oraciones): {lote_nlp.shape}")

# Extraemos la representación de la primera palabra de la primera oración
primera_palabra = lote_nlp[0, 0, :]
print(f"Vector embedding de la primera palabra: forma {primera_palabra.shape}")

### 4.3. Machine Learning (Regularización)
*   **Regularización L2 (Ridge):** Añade $\lambda \|\mathbf{w}\|_2^2$ a la función de pérdida. Favorece pesos pequeños.
*   **Regularización L1 (Lasso):** Añade $\lambda \|\mathbf{w}\|_1$. Favorece pesos dispersos (muchos ceros).

In [ ]:
# Simulamos los pesos de un modelo lineal (10 características, 5 neuronas)
W = np.random.randn(10, 5)
lambda_reg = 0.1

# Término de regularización L2 (suma de cuadrados)
l2_term = lambda_reg * np.sum(W**2)
print(f"📉 Término de regularización L2 (Ridge): {l2_term:.4f}")

# Término de regularización L1 (suma de valores absolutos)
l1_term = lambda_reg * np.sum(np.abs(W))
print(f"📈 Término de regularización L1 (Lasso): {l1_term:.4f}")

# Efecto en la dispersión: Contamos cuántos pesos son casi cero (<0.01)
num_ceros_l2 = np.sum(np.abs(W) < 0.01)
print(f"Número de pesos casi cero en W: {num_ceros_l2} de {W.size} (sin regularización, solo inicialización)")

---
## 5. Ejercicios para el Estudiante

1.  **Manipulación Creativa:** Crea un tensor 3D de forma (3, 5, 4) con números aleatorios. Luego:
    *   Transpónelo para que la forma sea (4, 5, 3).
    *   Rediménsionalo (reshape) a una matriz de 2 dimensiones que tenga el mismo número de elementos.
    *   Extrae la "rebanada" que contiene la segunda "fila" de la primera matriz.

2.  **Cálculo de Normas:** Dado el vector `v = np.array([1, -2, 3, -4])`:
    *   Calcula su norma L2, L1 y L-infinito usando NumPy.
    *   Verifica el resultado de la norma L1 manualmente (sumando valores absolutos).

3.  **Aplicación en Regularización:** Para una matriz de pesos `W` de forma (8, 4) (8 características, 4 neuronas), calcula el término de regularización L2 (asumiendo $\lambda = 0.05$) y L1. ¿Cuál de los dos términos es mayor y por qué?

4.  **Reflexión de NLP:** ¿Por qué crees que es necesario que un lote de oraciones tenga una dimensión de 'secuencia' fija (lo que a menudo requiere padding) si las oraciones en el mundo real tienen longitudes diferentes? ¿Cómo crees que se representa una oración más corta que las demás en el tensor 3D?